In [ ]:
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, applications
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

# 1. SETTINGS
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
warnings.filterwarnings('ignore')
data_path = r'Bone Fracture' 
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

# 2. DATA PREP
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2, rotation_range=15, zoom_range=0.1)
train_gen = train_datagen.flow_from_directory(data_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', subset='training')
val_gen = train_datagen.flow_from_directory(data_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', subset='validation', shuffle=False)

# 3. MODEL 2 ARCHITECTURE
base_model = applications.MobileNetV2(input_shape=(150, 150, 3), include_top=False, weights='imagenet')
base_model.trainable = False 
model_2 = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])
model_2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 4. CALLBACKS
checkpoint = ModelCheckpoint('best_bone_model.keras', save_best_only=True, monitor='val_loss')
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 5. TRAINING
print("🚀 Starting Training...")
history = model_2.fit(
    train_gen, 
    epochs=1, # Set to 100 for the final run!
    validation_data=val_gen,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

# 6. SAVE
model_2.save('final_model_2_fracture.keras')
print("✅ TRAINING COMPLETE. Model saved.")

# --- 7. THE ADVANCED REPORT FUNCTION ---
def generate_advanced_report(model, history, val_gen):
    print("📊 Generating Level 4 Dashboard...")
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(2, 2)
    
    # Accuracy Graph
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(history.history['accuracy'], label='Train Acc', color='#2ecc71')
    ax1.plot(history.history['val_accuracy'], label='Val Acc', color='#27ae60')
    ax1.set_title('Accuracy (Optimal Epoch Finder)')
    ax1.legend()

    # Loss Graph
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(history.history['loss'], label='Train Loss', color='#e74c3c')
    ax2.plot(history.history['val_loss'], label='Val Loss', color='#c0392b')
    ax2.set_title('Loss (Error Rate)')
    ax2.legend()

    # Metrics Calculation
    val_gen.reset()
    preds_prob = model.predict(val_gen, verbose=0)
    y_pred = (preds_prob > 0.5).astype(int).reshape(-1)
    y_true = val_gen.classes
    
    cm = confusion_matrix(y_true, y_pred)
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    # Confusion Matrix
    ax3 = fig.add_subplot(gs[1, 0])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax3, xticklabels=['Normal', 'Fracture'], yticklabels=['Normal', 'Fracture'])
    ax3.set_title('Confusion Matrix')

    # Metric Card
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.axis('off')
    stats = f"Level 4 Metrics\n\nPrecision: {pre:.1%}\nRecall: {rec:.1%}\nF1-Score: {f1:.1%}"
    ax4.text(0.5, 0.5, stats, ha='center', va='center', fontsize=20, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=1'))

    plt.tight_layout()
    plt.show()

# 8. CALL THE FUNCTION
generate_advanced_report(model_2, history, val_gen)